In [ ]:
%%writefile matmul_compare.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 1024
#define TILE_WIDTH 16

__global__ void matMulNaive(int A[][N], int B[][N], int C[][N], int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    int sum = 0;

    for (int k = 0; k < n; k++) {
        sum += A[row][k] * B[k][col];
    }

    C[row][col] = sum;
}

__global__ void matMulShared(int A[][N], int B[][N], int C[][N], int n) {
    __shared__ int tileA[TILE_WIDTH][TILE_WIDTH];
    __shared__ int tileB[TILE_WIDTH][TILE_WIDTH];

    int row = blockIdx.y * TILE_WIDTH + threadIdx.y;
    int col = blockIdx.x * TILE_WIDTH + threadIdx.x;

    int sum = 0;

    for (int t = 0; t < n / TILE_WIDTH; t++) {
        tileA[threadIdx.y][threadIdx.x] = A[row][t * TILE_WIDTH + threadIdx.x];
        tileB[threadIdx.y][threadIdx.x] = B[t * TILE_WIDTH + threadIdx.y][col];

        __syncthreads();

        for (int k = 0; k < TILE_WIDTH; k++) {
            sum += tileA[threadIdx.y][k] * tileB[k][threadIdx.x];
        }

        __syncthreads();
    }

    C[row][col] = sum;
}

void matMulCPU(int A[][N], int B[][N], int C[][N], int n) {
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            int sum = 0;
            for (int k = 0; k < n; k++) {
                sum += A[i][k] * B[k][j];
            }
            C[i][j] = sum;
        }
    }
}

void random_ints(int a[][N], int n) {
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            a[i][j] = (rand() % 10) + 1;
        }
    }
}

void print_corner(int a[][N], const char *label) {
    printf("%s\n", label);
    for (int i = 0; i < 5; i++) {
        for (int j = 0; j < 5; j++) printf("%d ", a[i][j]);
        printf("\n");
    }
    printf("\n");
}

int main(void) {
    int (*A)[N], (*B)[N], (*C_cpu)[N], (*C_gpu_naive)[N], (*C_gpu)[N];
    int (*d_A)[N], (*d_B)[N], (*d_C_naive)[N], (*d_C)[N];
    int size = N * N * sizeof(int);

    srand(time(NULL));

    A = (int (*)[N])malloc(size); random_ints(A, N);
    B = (int (*)[N])malloc(size); random_ints(B, N);
    C_cpu = (int (*)[N])malloc(size);
    C_gpu_naive = (int (*)[N])malloc(size);
    C_gpu = (int (*)[N])malloc(size);

    clock_t cpu_start = clock();
    matMulCPU(A, B, C_cpu, N);
    clock_t cpu_end = clock();
    double cpu_time_ms = ((double)(cpu_end - cpu_start) / CLOCKS_PER_SEC) * 1000.0;

    cudaMalloc((void **)&d_A, size);
    cudaMalloc((void **)&d_B, size);
    cudaMalloc((void **)&d_C_naive, size);
    cudaMalloc((void **)&d_C, size);

    cudaMemcpy(d_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, size, cudaMemcpyHostToDevice);

    dim3 dimBlock(TILE_WIDTH, TILE_WIDTH);
    dim3 dimGrid(N / TILE_WIDTH, N / TILE_WIDTH);

    cudaEvent_t gpu_naive_start, gpu_naive_stop;
    cudaEventCreate(&gpu_naive_start);
    cudaEventCreate(&gpu_naive_stop);

    cudaEventRecord(gpu_naive_start);
    matMulNaive<<<dimGrid, dimBlock>>>(d_A, d_B, d_C_naive, N);
    cudaEventRecord(gpu_naive_stop);
    cudaEventSynchronize(gpu_naive_stop);

    float gpu_naive_time_ms = 0;
    cudaEventElapsedTime(&gpu_naive_time_ms, gpu_naive_start, gpu_naive_stop);

    cudaMemcpy(C_gpu_naive, d_C_naive, size, cudaMemcpyDeviceToHost);

    cudaEvent_t gpu_start, gpu_stop;
    cudaEventCreate(&gpu_start);
    cudaEventCreate(&gpu_stop);

    cudaEventRecord(gpu_start);
    matMulShared<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, N);
    cudaEventRecord(gpu_stop);
    cudaEventSynchronize(gpu_stop);

    float gpu_time_ms = 0;
    cudaEventElapsedTime(&gpu_time_ms, gpu_start, gpu_stop);

    cudaMemcpy(C_gpu, d_C, size, cudaMemcpyDeviceToHost);

    printf("Matrix size: %d x %d\n", N, N);
    printf("Tile size: %d x %d\n", TILE_WIDTH, TILE_WIDTH);
    printf("Threads per block: %d x %d (%d total)\n", dimBlock.x, dimBlock.y, dimBlock.x * dimBlock.y);
    printf("Grid dimensions: %d x %d\n", dimGrid.x, dimGrid.y);
    printf("Total blocks: %d\n\n", dimGrid.x * dimGrid.y);

    print_corner(A, "Top-left 5x5 of A:");
    print_corner(B, "Top-left 5x5 of B:");
    print_corner(C_cpu, "Top-left 5x5 of C (CPU result):");
    print_corner(C_gpu_naive, "Top-left 5x5 of C (GPU no-shared result):");
    print_corner(C_gpu, "Top-left 5x5 of C (GPU result):");

    printf("Unparallelized (CPU) execution time: %f ms\n", cpu_time_ms);
    printf("Without shared memory (GPU) execution time: %f ms\n", gpu_naive_time_ms);
    printf("Shared memory (GPU) execution time: %f ms\n", gpu_time_ms);
    printf("Speedup (unparallelized / GPU with shared memory): %f\n", cpu_time_ms / gpu_time_ms);
    printf("Speedup (GPU without shared memory / GPU with shared memory): %f\n", gpu_naive_time_ms / gpu_time_ms);

    free(A); free(B); free(C_cpu); free(C_gpu_naive); free(C_gpu);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C_naive); cudaFree(d_C);
    cudaEventDestroy(gpu_naive_start);
    cudaEventDestroy(gpu_naive_stop);
    cudaEventDestroy(gpu_start);
    cudaEventDestroy(gpu_stop);

    return 0;
}

Overwriting matmul_compare.cu


In [ ]:
!nvcc -arch=sm_75 matmul_compare.cu -o matmul_compare

In [ ]:
!./matmul_compare

Matrix size: 1024 x 1024
Tile size: 16 x 16
Threads per block: 16 x 16 (256 total)
Grid dimensions: 64 x 64
Total blocks: 4096

Top-left 5x5 of A:
1 2 3 4 6 
3 9 10 9 5 
8 6 3 6 2 
1 6 8 6 7 
5 6 1 9 5 

Top-left 5x5 of B:
6 6 3 6 3 
4 3 2 10 2 
8 9 3 6 2 
1 10 9 1 10 
4 4 8 7 4 

Top-left 5x5 of C (CPU result):
31546 30955 30746 30645 31677 
31666 32467 31104 30994 32238 
31394 30761 31159 30291 31526 
31952 31311 31133 31241 31808 
30037 29942 29546 29240 30064 

Top-left 5x5 of C (GPU no-shared result):
31546 30955 30746 30645 31677 
31666 32467 31104 30994 32238 
31394 30761 31159 30291 31526 
31952 31311 31133 31241 31808 
30037 29942 29546 29240 30064 

Top-left 5x5 of C (GPU result):
31546 30955 30746 30645 31677 
31666 32467 31104 30994 32238 
31394 30761 31159 30291 31526 
31952 31311 31133 31241 31808 
30037 29942 29546 29240 30064 

Unparallelized (CPU) execution time: 6199.079000 ms
Without shared memory (GPU) execution time: 4.912224 ms
Shared memory (GPU) execution time: 